In [ ]:
!pip install google-cloud-translate

In [ ]:
api_key = "###"

In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import re
from tqdm import tqdm
import seaborn as sns
import matplotlib.pyplot as plt
import torch
from sklearn.manifold import TSNE
from tqdm import tqdm
import plotly.express as px
import requests
import time

In [ ]:
# mount google drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd

# Load the dataset
pm_india_df = pd.read_csv("/content/drive/MyDrive/News_Analysis_Data/PM-all-speech/combined_translations.csv")
pm_india_df.head()

# Translations using Google Translate for all the language which are not English



In [ ]:
tqdm.pandas()

# Translate using Google Cloud Translation API
def translate_to_english(text, source_lang, api_key):
    try:
        if not isinstance(text, str) or text.strip() == "":
            return None

        url = "https://translation.googleapis.com/language/translate/v2"
        params = {
            "q": text,
            "source": source_lang,
            "target": "en",
            "key": api_key,
            "format": "text"
        }

        response = requests.post(url, data=params)
        if response.status_code == 200:
            return response.json()["data"]["translations"][0]["translatedText"]
        else:
            print(f"Error: {response.status_code} - {response.text}")
            return None
    except Exception as e:
        print(f"Error translating from {source_lang}: {text[:50]}..., Error: {e}")
        return None

In [ ]:
# Create a new DataFrame and copy original English column
pm_ind_translated_df = pd.DataFrame()
pm_ind_translated_df["original_english"] = pm_india_df["English"]

In [ ]:
# Google Translate language codes
lang_map = {
    "meitei": "mni-Mtei", "oriya": "or", "hindi": "hi", "tamil": "ta", "bengali": "bn", "urdu": "ur",
    "assamese": "as", "marathi": "mr", "telugu": "te", "gujarati": "gu", "punjabi": "pa",
    "kannada": "kn", "malayalam": "ml"
}

# Translate each language column using Google Translate API
for lang, code in lang_map.items():
    target_col = f"{lang}_en_translation"
    print(f"Translating from {lang} ({code}) to English...")
    pm_ind_translated_df[target_col] = pm_india_df[lang].progress_apply(lambda x: translate_to_english(x, code, api_key))
    time.sleep(1)  


In [ ]:
# save translated_DF
pm_ind_translated_df.to_csv("/content/drive/MyDrive/News_Analysis_Data/translated_PM_INDIA_WITH_GOOGLE_API.csv", index=True)

In [ ]:
pm_ind_translated_df = pd.read_csv("/content/drive/MyDrive/News_Analysis_Data/translated_PM_INDIA_WITH_GOOGLE_API.csv")

# Geenrate Embedddings using MPNet


In [ ]:
# check device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2").to(device)


In [ ]:
pm_ind_translated_df.head()

In [ ]:
# remove unamed coloum
pm_ind_translated_df = pm_ind_translated_df.loc[:, ~pm_ind_translated_df.columns.str.contains('^Unnamed')]

In [ ]:
# Store all row-wise embeddings in a list
pm_ind_embeddings = []

# Generate embeddings row-wise
for _, row in tqdm(pm_ind_translated_df.iterrows(), total=pm_ind_translated_df.shape[0], desc="Generating embeddings"):
    sentence_list = row.astype(str).tolist() 
    embeddings = model.encode(sentence_list)
    pm_ind_embeddings.append(embeddings)  # shape: [num_languages, embedding_dim]


In [ ]:
# create a DataFrame with same shape as original to store embeddings
pm_ind_embeddings_df = pd.DataFrame(index=pm_ind_translated_df.index, columns=pm_ind_translated_df.columns)

# fill the DataFrame using the precomputed embeddings
for row_idx, row_embeddings in enumerate(pm_ind_embeddings):
    for col_idx, lang in enumerate(pm_ind_translated_df.columns):
        pm_ind_embeddings_df.at[row_idx, lang] = row_embeddings[col_idx]

In [ ]:
pm_ind_embeddings_df.head()


# t-SNE Dimesionality Reduction

In [ ]:
def prepare_tsne_data(df):
    data = []
    labels = []
    row_indices = []

    for row_idx, row in df.iterrows():
        for lang in df.columns:
            embedding = row[lang]
            data.append(embedding)
            labels.append(lang)
            row_indices.append(row_idx)

    return np.array(data), labels, row_indices

In [ ]:
# Prepare
all_embeddings, languages, row_ids = prepare_tsne_data(pm_ind_embeddings_df)

tsne = TSNE(
    n_components=2,
    perplexity=5,
    max_iter=2000,
    learning_rate=100,
    metric="cosine",
    random_state=42
)

tsne_result = tsne.fit_transform(all_embeddings)

# Make DataFrame
tsne_df = pd.DataFrame({
    'x': tsne_result[:, 0],
    'y': tsne_result[:, 1],
    'Language': languages,
    'Row': row_ids
})


In [ ]:
fig = px.scatter(
    tsne_df,
    x='x',
    y='y',
    color='Row',
    hover_data=['Language', 'Row'],
    title="t-SNE of Multilingual Sentence Embeddings by MPNet (Colored by Row)",

)
fig.update_traces(marker=dict(size=6, opacity=0.8))
fig.show()

In [ ]:

fig = px.scatter(
    tsne_df,
    x='x',
    y='y',
    color='Language',
    hover_data=['Language', 'Row'],
    title="Global t-SNE of Multilingual Sentence Embeddings using MPNet (Colored by Language)",

)
fig.update_traces(marker=dict(size=6, opacity=0.8))
fig.show()

In [ ]:
# Filter only Row 0
row_0_df = tsne_df[tsne_df["Row"] == 0]

# Plot
fig = px.scatter(
    row_0_df,
    x='x',
    y='y',
    color='Language',
    hover_data=['Language'],
    title='t-SNE Semantic Map for Row 0 (Same Sentence Across Languages)',

)

fig.update_traces(marker=dict(size=10, opacity=0.9))
fig.update_layout(showlegend=True)
fig.show()


# Cosine Similarity with original 768-D embeddings

In [ ]:
similarity_matrices = []  # to hold similarity matrix per sentence

for idx, row in pm_ind_embeddings_df.iterrows():
    embeddings = np.array(row.tolist())  # shape: [num_languages, embedding_dim]
    sim_matrix = cosine_similarity(embeddings)  # shape: [num_langs x num_langs]
    similarity_matrices.append(sim_matrix)

In [ ]:
# Stack all matrices: shape becomes [num_rows, num_langs, num_langs]
similarity_stack = np.stack(similarity_matrices)

# Mean similarity matrix across all rows
avg_similarity_matrix = np.mean(similarity_stack, axis=0)


In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(avg_similarity_matrix, annot=True, fmt=".2f",
            xticklabels=pm_ind_embeddings_df.columns.tolist(),
            yticklabels=pm_ind_embeddings_df.columns.tolist(),
            cmap="YlGnBu")
plt.title("Average Cosine Similarity Matrix Across All Sentences")
plt.tight_layout()
plt.show()


# Cosine Similarity on t-SNE reduced embeddings

In [ ]:
tsne_wide_df = (
    tsne_df
    .assign(coords=lambda df: df[['x', 'y']].values.tolist())
    .pivot(index='Row', columns='Language', values='coords')
)


In [ ]:
tsne_wide_df.head()

In [ ]:
similarity_matrices_tsne = []  # to hold similarity matrix per sentence

for idx, row in tsne_wide_df.iterrows():
    embeddings = np.array(row.tolist())
    sim_matrix = cosine_similarity(embeddings)
    similarity_matrices_tsne.append(sim_matrix)

In [ ]:
# Stack all matrices: shape becomes [num_rows, num_langs, num_langs]
similarity_stack_tsne = np.stack(similarity_matrices_tsne)

# Mean similarity matrix across all rows
avg_similarity_matrix_tsne = np.mean(similarity_stack_tsne, axis=0)


In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(avg_similarity_matrix_tsne, annot=True, fmt=".2f",
            xticklabels=tsne_wide_df.columns.tolist(),
            yticklabels=tsne_wide_df.columns.tolist(),
            cmap="YlGnBu")
plt.title("Average Cosine Similarity Matrix Across All Sentences with t-SNE reduced embeddings")
plt.tight_layout()
plt.show()